# 01 — NR-IQA benchmark on UID2021 (O1 + O2)

**Before running, in the right-hand panel → Settings:**
- Accelerator: **GPU T4 x2** or **GPU P100**
- Internet: **On** (needed for `pip`, the GitHub clone, pyiqa weights and the dataset download)

**Persistence:** `/kaggle/working` is only kept if you use **Save Version → Save & Run All (Commit)**.
For the long scoring step, commit the notebook rather than running interactively. To resume after a timeout,
add the previous version's output as an input and copy `results/scores/*.csv` back (cell 3b): the scorer skips work already done.

In [ ]:
# 1. Code
!git clone -q https://github.com/Banasri007/UW-detection-aware-iqa.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -e ".[iqa,dev]"
!python -m pytest -q 2>&1 | tail -2

In [ ]:
# 2. Paths. Inputs on Kaggle are read-only, so data we download and all outputs go to /kaggle/working.
import os
os.environ["DATA_ROOT"] = "/kaggle/working/data"
os.environ["RESULTS_ROOT"] = "/kaggle/working/results"
# If you instead attached UID2021 as a Kaggle Dataset, point straight at it:
# os.environ["UID2021_ROOT"] = "/kaggle/input/<your-uid2021-slug>"
os.makedirs(os.environ["DATA_ROOT"], exist_ok=True)

In [ ]:
# 3. Download UID2021 from the authors' Google Drive link (skip if you set UID2021_ROOT above).
#    Licence: non-commercial research use; cite Hou et al., ACM TOMM 2023.
import os, glob, shutil, subprocess
root = "/kaggle/working/data/UID2021"
if "UID2021_ROOT" not in os.environ and not os.path.exists(root):
    !pip install -q gdown
    os.makedirs(root, exist_ok=True)
    !gdown -q --fuzzy "https://drive.google.com/file/d/1cD8a_IsQHkQ_q-mDV6PvRql_gTZPRnk5/view" -O /kaggle/working/uid2021_archive
    kind = subprocess.run(["file", "-b", "/kaggle/working/uid2021_archive"], capture_output=True, text=True).stdout
    print("archive type:", kind)
    if "Zip" in kind:
        shutil.unpack_archive("/kaggle/working/uid2021_archive", root, "zip")
    else:  # RAR / 7z
        !7z x -y -o{root} /kaggle/working/uid2021_archive > /dev/null
    os.remove("/kaggle/working/uid2021_archive")
print(os.environ.get("UID2021_ROOT", root))

In [ ]:
# 3b. (Resume only) copy scores from a previous committed version attached as input.
# !mkdir -p /kaggle/working/results/scores && cp /kaggle/input/<prev-version>/results/scores/*.csv /kaggle/working/results/scores/

In [ ]:
# 4. Look at what we got: image folders + every score table with its columns.
from uwiqa.data import dataset_root, inspect_dataset
inspect_dataset(dataset_root("uid2021"))

**5. Fill these three values in from the output above** (path of the MOS table relative to the dataset root,
the column holding image names, the column holding MOS). Scene/method parsing from the `SI_B_1` / `ERI_B_1_HP`
naming is already configured.

In [ ]:
UID_TABLE = "TODO.xlsx"   # e.g. "UID2021/MOS.xlsx"
IMAGE_COL = "TODO"
MOS_COL   = "TODO"

import yaml
p = "configs/datasets.yaml"
cfg = yaml.safe_load(open(p))
cfg["uid2021"].update(table=UID_TABLE, image_col=IMAGE_COL, mos_col=MOS_COL)
yaml.safe_dump(cfg, open(p, "w"), sort_keys=False)
!python scripts/build_manifests.py --only uid2021
import pandas as pd
from uwiqa.data import load_manifest
m = load_manifest("uid2021"); print(m.method.value_counts()); m.head()

In [ ]:
# 6. Check every metric loads on the GPU (downloads ~1-2 GB of pretrained weights once).
!python scripts/check_env.py

In [ ]:
# 7. Score all 960 images with every metric (resumable). ~30-60 min on a T4.
!python scripts/score_metrics.py --dataset uid2021

In [ ]:
# 8. O1 table + Week-4 gate verdict.
!python scripts/benchmark.py --dataset uid2021

In [ ]:
# 9. O2 over-enhancement stress test on the 60 raw images.
!python scripts/stress_test.py --dataset uid2021 --metrics uiqm uciqe topiq_nr liqe musiq uranker

In [ ]:
# 10. Bundle results (also visible in the Output tab after a commit).
!cd /kaggle/working && zip -qr results.zip results && ls -lh results.zip